In [21]:
# Import & configs
import requests
import time
import json
import logging
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path
from google.cloud import storage

# ── GCP config ────────────────────────────────────────────────
GCP_PROJECT   = "vuthesis-llm-buzz"
GCS_BUCKET    = "thesis-bucket-vua"
BRONZE_PREFIX = "bronze/reddit"

# ── Scraper config ────────────────────────────────────────────
PRE_LAUNCH_DAYS   = 90        # window before launch date
POSTS_PER_REQUEST = 100       # Reddit max per call
SLEEP_BETWEEN_REQUESTS = 2    # seconds — respect rate limits
SLEEP_BETWEEN_EVENTS   = 10   # seconds — pause between products
MIN_POST_THRESHOLD = 500      # warn if event falls below this

# ── Paths ─────────────────────────────────────────────────────
EVENTS_PATH     = Path(r"C:\Users\User\Desktop\VU\Thesis\Code\LLM-prelaunch-buzz-postlaunch-predictor\events.csv")
CHECKPOINT_PATH = Path("checkpoint.json")
CACHE_DIR       = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("reddit_scraper.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

# ── Headers — browser-like UA avoids 429 blocks ───────────────
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
}

log.info("Config loaded")

2026-05-18 16:14:22,493 [INFO] Config loaded


In [22]:
events = pd.read_csv(EVENTS_PATH, encoding="utf-8")
events.columns = events.columns.str.strip()
events["launch_date"] = pd.to_datetime(
    events["launch_date"].str.strip(), dayfirst=True, utc=True
)
events["subreddits_list"] = [[] for _ in range(len(events))]

print(f"Loaded {len(events)} events:")
print("Columns:", events.columns.tolist())
display(events[["product_name", "launch_date", "product_type"]])

Loaded 35 events:
Columns: ['product_name', 'brand', 'launch_date', 'product_type', 'pre_launch_subreddits', 'subreddits_list']


,product_name,launch_date,product_type
0,Fire HD 10 (2019),2019-10-30 00:00:00+00:00,Utilitarian
1,Fire HD 8 (2020),2020-09-30 00:00:00+00:00,Utilitarian
2,Fire 7 (2019),2019-05-29 00:00:00+00:00,Utilitarian
3,iPad Pro 2020,2020-03-25 00:00:00+00:00,Hedonic
4,iPad Air 2 (2014),2014-10-16 00:00:00+00:00,Hedonic
5,MacBook Air M1,2020-11-17 00:00:00+00:00,Hedonic
6,iPad mini 6,2021-09-24 00:00:00+00:00,Hedonic
7,"MacBook Pro 16""",2021-06-10 00:00:00+00:00,Hedonic
8,Galaxy S23,2023-02-17 00:00:00+00:00,Utilitarian
9,Pixel 5,2020-10-15 00:00:00+00:00,Utilitarian


In [23]:
# Fix the product name in events DataFrame
for idx, row in events.iterrows():
    if "ipad pro" in row["product_name"].lower() and "m2" not in row["product_name"].lower() and "2020" not in row["product_name"].lower():
        events.at[idx, "product_name"] = "iPad Pro M2"
        events.at[idx, "subreddits_list"] = ["ipad", "iPadPro", "apple", "ios", "tablets", "technology"]
        print(f"Fixed row {idx}: {row['product_name']} -> iPad Pro M2")

Fixed row 18: iPad Pro  -> iPad Pro M2


In [ ]:
overrides = {
    "Fire HD 10":  ["kindlefire", "amazonecho", "amazon", "tablets",
                    "android", "gadgets", "technology"],
    "Fire HD 8":   ["kindlefire", "amazonecho", "amazon", "tablets",
                    "android", "gadgets", "technology"],
    "Fire 7":      ["kindlefire", "tablets", "android", "gadgets"],
    "iPad Pro 2020":  ["ipad", "iPadPro", "apple", "technology", "gadgets"],
    "iPad Air 2":     ["ipad", "apple", "technology", "gadgets"],
    "MacBook Air M1": ["apple", "mac", "macbook", "applesilicon", "technology", "laptops"],
    "iPad mini 6":    ["ipad", "iPadmini", "apple", "ios", "tablets", "technology"],
    "MacBook Pro 16": ["apple", "mac", "macbook", "applesilicon", "technology", "laptops"],
    "Galaxy S23":     ["GalaxyS23", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Pixel 5":        ["GooglePixel", "pixel5", "android", "technology", "gadgets"],
    "iPhone 14":      ["iphone", "apple", "ios", "technology", "gadgets", "smartphones"],
    "iPhone 15":      ["iphone", "apple", "ios", "technology", "gadgets", "smartphones"],
    "Galaxy S22":     ["GalaxyS22", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Pixel 7":        ["GooglePixel", "pixel7", "android", "technology", "gadgets", "smartphones"],
    "iPhone 13":      ["iphone", "apple", "ios", "technology", "gadgets", "smartphones"],
    "Galaxy S21":     ["GalaxyS21", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Pixel 6":        ["GooglePixel", "pixel6", "android", "technology", "gadgets"],
    "MacBook Air M2": ["apple", "mac", "macbook", "MacOS", "technology", "laptops"],
    "iPad Pro M2":    ["ipad", "iPadPro", "apple", "ios", "tablets", "technology"],
    "iPad Pro ":      ["ipad", "iPadPro", "apple", "ios", "tablets", "technology"],

    # Samsung Galaxy S series
    "Samsung Galaxy S4":       ["galaxys4", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S5":       ["galaxys5", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S6":       ["GalaxyS6", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S7 Edge":  ["GalaxyS7", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S8":       ["GalaxyS8", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S20 FE":   ["GalaxyS20FE", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S21 Ultra":["GalaxyS21", "samsung", "android", "technology", "gadgets", "smartphones"],
    "Samsung Galaxy S22 Ultra":["GalaxyS22", "samsung", "android", "technology", "gadgets", "smartphones"],

    # Google Pixel
    "Google Pixel 3a": ["GooglePixel", "pixel3a", "android", "technology", "gadgets", "smartphones"],
    "Google Pixel 4 XL": ["GooglePixel", "pixel4", "android", "technology", "gadgets", "smartphones"],
    "Google Pixel 4a": ["GooglePixel", "pixel4a", "android", "technology", "gadgets", "smartphones"],

    # Moto
    "Moto G 3rd Gen":   ["motorola", "android", "moto", "technology", "gadgets", "smartphones"],
    "Moto G 4th Gen":   ["motorola", "android", "moto", "technology", "gadgets", "smartphones"],
    "Moto G Fast":      ["motorola", "android", "moto", "technology", "gadgets", "smartphones"],
    "Moto G Power 2022":["motorola", "android", "moto", "technology", "gadgets", "smartphones"],

    # LG
    "LG G3": ["lgg3", "android", "technology", "gadgets", "smartphones"],


for product, subs in overrides.items():
    for idx, row in events.iterrows():
        if product.lower().strip() in row["product_name"].lower():
            events.at[idx, "subreddits_list"] = subs
            break

print("Subreddit assignment:")
for _, row in events.iterrows():
    print(f"  {row['product_name']}: {len(row['subreddits_list'])} subreddits — {row['subreddits_list']}")

Subreddit assignment:
  Fire HD 10 (2019): 7 subreddits — ['kindlefire', 'amazonecho', 'amazon', 'tablets', 'android', 'gadgets', 'technology']
  Fire HD 8 (2020) : 7 subreddits — ['kindlefire', 'amazonecho', 'amazon', 'tablets', 'android', 'gadgets', 'technology']
  Fire 7 (2019) : 4 subreddits — ['kindlefire', 'tablets', 'android', 'gadgets']
  iPad Pro 2020 : 6 subreddits — ['ipad', 'iPadPro', 'apple', 'ios', 'tablets', 'technology']
  iPad Air 2 (2014): 4 subreddits — ['ipad', 'apple', 'technology', 'gadgets']
  MacBook Air M1 : 6 subreddits — ['apple', 'mac', 'macbook', 'applesilicon', 'technology', 'laptops']
  iPad mini 6: 6 subreddits — ['ipad', 'iPadmini', 'apple', 'ios', 'tablets', 'technology']
  MacBook Pro 16": 6 subreddits — ['apple', 'mac', 'macbook', 'applesilicon', 'technology', 'laptops']
  Galaxy S23: 6 subreddits — ['GalaxyS23', 'samsung', 'android', 'technology', 'gadgets', 'smartphones']
  Pixel 5: 5 subreddits — ['GooglePixel', 'pixel5', 'android', 'technology', 

In [26]:
#Checkpoint helpers - helps track which events are fully scrapped 
# so it can be resumed if interrupted
def load_checkpoint() -> dict:
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(checkpoint: dict):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(checkpoint, f, indent=2)

def mark_complete(checkpoint: dict, event_key: str, post_count: int):
    checkpoint[event_key] = {
        "status": "complete",
        "post_count": post_count,
        "completed_at": datetime.utcnow().isoformat()
    }
    save_checkpoint(checkpoint)

checkpoint = load_checkpoint()
print(f"Checkpoint loaded — {len(checkpoint)} events already completed:")
for k, v in checkpoint.items():
    print(f"  {k}: {v['post_count']} posts ({v['completed_at']})")

Checkpoint loaded — 19 events already completed:
  galaxy_s23: 984 posts (2026-05-08T12:52:50.135549)
  iphone_15: 670 posts (2026-05-08T13:02:33.114515)
  pixel_7: 751 posts (2026-05-08T13:12:40.701465)
  ipad_pro_2020_: 2834 posts (2026-05-11T12:20:30.971554)
  macbook_air_m1: 1307 posts (2026-05-11T12:21:38.902835)
  ipad_mini_6: 1007 posts (2026-05-11T12:22:30.712038)
  iphone_14: 15186 posts (2026-05-11T12:37:01.315668)
  iphone_13: 9561 posts (2026-05-11T12:42:42.529582)
  galaxy_s21: 4079 posts (2026-05-11T12:45:50.475036)
  pixel_6: 40470 posts (2026-05-11T13:08:42.625164)
  macbook_air_m2: 3410 posts (2026-05-11T13:11:03.398158)
  fire_7_(2019)_: 5 posts (2026-05-12T15:45:54.316860)
  ipad_air_2_(2014): 5 posts (2026-05-12T15:46:13.165740)
  macbook_air_m1_: 1061 posts (2026-05-12T15:47:32.006522)
  macbook_pro_16": 1026 posts (2026-05-12T15:49:21.705679)
  pixel_5: 21112 posts (2026-05-12T16:07:20.281323)
  ipad_pro_m2: 1598 posts (2026-05-12T16:08:31.815997)
  fire_hd_10_(20

In [27]:
# Core scrapping functions
def fetch_posts_page(subreddit: str, query: str, after: str = None) -> dict | None:
    """Fetch one page (up to 100 posts) from Reddit search JSON endpoint."""
    url = f"https://www.reddit.com/r/{subreddit}/search.json"
    params = {
        "q":           query,
        "restrict_sr": 1,
        "sort":        "new",
        "t":           "all",
        "limit":       POSTS_PER_REQUEST,
    }
    if after:
        params["after"] = after

    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
        if resp.status_code == 429:
            log.warning("Rate limited — sleeping 60s")
            time.sleep(60)
            return fetch_posts_page(subreddit, query, after)  # retry once
        if resp.status_code != 200:
            log.error(f"HTTP {resp.status_code} for r/{subreddit} query='{query}'")
            return None
        return resp.json()["data"]
    except Exception as e:
        log.error(f"Request error: {e}")
        return None


def fetch_top_comments(subreddit: str, post_id: str) -> list[dict]:
    """Fetch top-level comments for a single post."""
    url = f"https://www.reddit.com/r/{subreddit}/comments/{post_id}.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return []
        data = resp.json()
        # data[1] contains comment listing
        comments = data[1]["data"]["children"]
        return [
            c["data"] for c in comments
            if c["kind"] == "t1"  # t1 = comment
            and not c["data"].get("body") in ["[deleted]", "[removed]", None]
        ]
    except Exception as e:
        log.warning(f"Could not fetch comments for {post_id}: {e}")
        return []


def parse_post(raw: dict, product_event: str, launch_ts: float) -> dict | None:
    """Extract relevant fields from a raw Reddit post dict."""
    body = raw.get("selftext", "") or ""
    title = raw.get("title", "") or ""

    # Skip deleted/removed
    if body in ["[deleted]", "[removed]"] or not title:
        return None

    created_utc = raw.get("created_utc", 0)
    days_to_launch = (launch_ts - created_utc) / 86400  # positive = before launch

    return {
        "post_id":       raw.get("id"),
        "product_event": product_event,
        "subreddit":     raw.get("subreddit"),
        "title":         title,
        "body_text":     body,
        "upvotes":       raw.get("score", 0),
        "num_comments":  raw.get("num_comments", 0),
        "created_utc":   created_utc,
        "days_to_launch": round(days_to_launch, 2),
        "url":           f"https://reddit.com{raw.get('permalink', '')}",
        "type":          "post",
    }


def parse_comment(raw: dict, product_event: str, post_id: str,
                  subreddit: str, launch_ts: float) -> dict | None:
    """Extract relevant fields from a raw Reddit comment dict."""
    body = raw.get("body", "") or ""
    if body in ["[deleted]", "[removed]"] or not body.strip():
        return None

    created_utc = raw.get("created_utc", 0)
    days_to_launch = (launch_ts - created_utc) / 86400

    return {
        "post_id":        f"comment_{raw.get('id')}",
        "parent_post_id": post_id,
        "product_event":  product_event,
        "subreddit":      subreddit,
        "title":          "",
        "body_text":      body,
        "upvotes":        raw.get("score", 0),
        "num_comments":   0,
        "created_utc":    created_utc,
        "days_to_launch": round(days_to_launch, 2),
        "url":            f"https://reddit.com{raw.get('permalink', '')}",
        "type":           "comment",
    }

log.info("Core functions defined")

2026-05-18 16:17:53,764 [INFO] Core functions defined


In [28]:
# GCS Upload helper
gcs_client = storage.Client(project=GCP_PROJECT)
gcs_bucket = gcs_client.bucket(GCS_BUCKET)

def upload_batch_to_gcs(records: list[dict], product_event: str) -> str:
    """Upload a list of records as a JSONL file to GCS Bronze.
    Returns the GCS path of the uploaded file.
    """
    if not records:
        return ""

    timestamp = int(time.time())
    gcs_path = f"{BRONZE_PREFIX}/{product_event}/{timestamp}.jsonl"
    blob = gcs_bucket.blob(gcs_path)

    jsonl_content = "\n".join(json.dumps(r) for r in records)

    blob.upload_from_string(
        jsonl_content,
        content_type="application/jsonl",
        retry=storage.retry.DEFAULT_RETRY  # resumable — survives drops
    )
    log.info(f"  Uploaded {len(records)} records → gs://{GCS_BUCKET}/{gcs_path}")
    return gcs_path


def upload_events_csv():
    """Back up events.csv to GCS config folder."""
    blob = gcs_bucket.blob("config/events.csv")
    blob.upload_from_filename(str(EVENTS_PATH))
    log.info("events.csv backed up to gs://config/events.csv")

upload_events_csv()
log.info("GCS client ready")

2026-05-18 16:18:03,500 [INFO] events.csv backed up to gs://config/events.csv
2026-05-18 16:18:03,502 [INFO] GCS client ready


In [30]:
#Single event scrapper
def scrape_event(row: pd.Series, fetch_comments: bool = True,
                 batch_size: int = 200) -> int:
    """
    Scrape all pre-launch posts (and optionally comments) for one product event.
    Uploads in batches of `batch_size` records to GCS.
    Returns total number of records collected.
    """
    product_event = row["product_name"].lower().replace(" ", "_")
    launch_dt     = row["launch_date"]
    launch_ts     = launch_dt.timestamp()
    window_start  = launch_dt - timedelta(days=PRE_LAUNCH_DAYS)
    window_start_ts = window_start.timestamp()
    subreddits    = row["subreddits_list"]
    query         = row["product_name"]  # use product name as search query

    log.info(f"\n{'='*60}")
    log.info(f"Scraping: {product_event}")
    log.info(f"Window: {window_start.date()} → {launch_dt.date()}")
    log.info(f"Subreddits: {subreddits}")

    all_records = []
    seen_post_ids = set()
    total_uploaded = 0

    for subreddit in subreddits:
        log.info(f"  → r/{subreddit}")
        after = None
        sub_post_count = 0
        out_of_window = False

        while not out_of_window:
            page = fetch_posts_page(subreddit, query, after)
            if not page or not page.get("children"):
                break

            for child in page["children"]:
                if child["kind"] != "t3":  # t3 = post
                    continue
                raw = child["data"]
                created_utc = raw.get("created_utc", 0)

                # Stop if we've gone past the 90-day window
                if created_utc < window_start_ts:
                    out_of_window = True
                    break

                # Skip if post is after launch date
                if created_utc >= launch_ts:
                    continue

                post_id = raw.get("id")
                if post_id in seen_post_ids:
                    continue
                seen_post_ids.add(post_id)

                parsed = parse_post(raw, product_event, launch_ts)
                if parsed:
                    all_records.append(parsed)
                    sub_post_count += 1

                    # Fetch top-level comments for this post
                    if fetch_comments and raw.get("num_comments", 0) > 0:
                        time.sleep(0.5)  # brief pause before comment fetch
                        comments = fetch_top_comments(subreddit, post_id)
                        for c in comments:
                            parsed_c = parse_comment(
                                c, product_event, post_id, subreddit, launch_ts
                            )
                            if parsed_c:
                                all_records.append(parsed_c)

                # Upload in batches to avoid memory buildup
                if len(all_records) >= batch_size:
                    upload_batch_to_gcs(all_records, product_event)
                    total_uploaded += len(all_records)
                    all_records = []

            after = page.get("after")
            if not after:
                break

            time.sleep(SLEEP_BETWEEN_REQUESTS)

        log.info(f"     r/{subreddit}: {sub_post_count} posts in window")

    # Upload any remaining records
    if all_records:
        upload_batch_to_gcs(all_records, product_event)
        total_uploaded += len(all_records)

    log.info(f"  TOTAL records uploaded for {product_event}: {total_uploaded}")
    if total_uploaded < MIN_POST_THRESHOLD:
        log.warning(
            f"  ⚠ {product_event} has {total_uploaded} records — "
            f"below minimum threshold of {MIN_POST_THRESHOLD}. "
            f"Consider adding more subreddits or broadening the search query."
        )

    return total_uploaded

log.info("scrape_event() defined")

2026-05-18 16:19:29,256 [INFO] scrape_event() defined


In [87]:
# Cell 6b - Arctic Shift scraper with multi-query support
def scrape_event_arctic_shift(row, fetch_comments=True):
    product_event  = row["product_name"].lower().replace(" ", "_")
    launch_dt      = row["launch_date"]
    window_start   = launch_dt - timedelta(days=PRE_LAUNCH_DAYS)
    subreddits     = row["subreddits_list"]

    # Support single string or list of queries
    raw_query = row.get("query_list", row["product_name"])
    queries   = raw_query if isinstance(raw_query, list) else [raw_query]

    after_ts  = int(window_start.timestamp())
    before_ts = int(launch_dt.timestamp())
    after_str = window_start.strftime("%Y-%m-%d")

    log.info(f"\n{'='*60}")
    log.info(f"Scraping (Arctic Shift): {product_event}")
    log.info(f"Window: {window_start.date()} to {launch_dt.date()}")
    log.info(f"Queries: {queries}")
    log.info(f"Subreddits: {subreddits}")

    seen_post_ids = set()  # dedup across all queries + subreddits
    total_uploaded = 0

    for query in queries:
        log.info(f"\n  Query: '{query}'")
        query_records = []

        for subreddit in subreddits:
            log.info(f"  -> r/{subreddit}")
            sub_post_count = 0
            before_str_cur = launch_dt.strftime("%Y-%m-%d")

            while True:
                url = "https://arctic-shift.photon-reddit.com/api/posts/search"
                params = {
                    "subreddit": subreddit,
                    "title":     query,
                    "after":     after_str,
                    "before":    before_str_cur,
                    "limit":     100,
                    "sort":      "desc",
                }

                try:
                    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
                    if resp.status_code in (429, 422):
                        wait = 60 if resp.status_code == 422 else 30
                        log.warning(f"Rate limited ({resp.status_code}) -- sleeping {wait}s")
                        time.sleep(wait)
                        continue
                    if resp.status_code != 200:
                        log.error(f"HTTP {resp.status_code}: {resp.text[:200]}")
                        break
                    posts = resp.json().get("data", [])
                except Exception as e:
                    log.error(f"Request error: {e}")
                    break

                if not posts:
                    break

                for post in posts:
                    post_id = post.get("id")
                    if not post_id or post_id in seen_post_ids:
                        continue
                    seen_post_ids.add(post_id)

                    parsed = {
                        "post_id":        post_id,
                        "product_event":  product_event,
                        "subreddit":      post.get("subreddit"),
                        "title":          post.get("title", ""),
                        "body_text":      post.get("selftext", ""),
                        "upvotes":        post.get("score", 0),
                        "num_comments":   post.get("num_comments", 0),
                        "created_utc":    post.get("created_utc"),
                        "days_to_launch": round(
                            (before_ts - int(post.get("created_utc", before_ts))) / 86400, 2
                        ),
                        "url":            f"https://reddit.com{post.get('permalink', '')}",
                        "type":           "post",
                        "query_matched":  query,
                    }

                    if parsed["body_text"] in ["[deleted]", "[removed]"] and not parsed["title"]:
                        continue

                    query_records.append(parsed)
                    sub_post_count += 1

                    if sub_post_count % 100 == 0:
                        log.info(f"     ... {sub_post_count} posts so far in r/{subreddit}")

                    # Fetch top-level comments
                    if fetch_comments and parsed["num_comments"] > 0:
                        time.sleep(0.5)
                        try:
                            c_resp = requests.get(
                                "https://arctic-shift.photon-reddit.com/api/comments/search",
                                headers=HEADERS,
                                params={"link_id": f"t3_{post_id}", "limit": 100},
                                timeout=30
                            )
                            if c_resp.status_code == 200:
                                for c in c_resp.json().get("data", []):
                                    body = c.get("body", "")
                                    if body in ["[deleted]", "[removed]", ""]:
                                        continue
                                    c_id = c.get("id")
                                    if c_id and c_id not in seen_post_ids:
                                        seen_post_ids.add(c_id)
                                        query_records.append({
                                            "post_id":        f"comment_{c_id}",
                                            "parent_post_id": post_id,
                                            "product_event":  product_event,
                                            "subreddit":      subreddit,
                                            "title":          "",
                                            "body_text":      body,
                                            "upvotes":        c.get("score", 0),
                                            "num_comments":   0,
                                            "created_utc":    c.get("created_utc"),
                                            "days_to_launch": round(
                                                (before_ts - int(c.get("created_utc", before_ts))) / 86400, 2
                                            ),
                                            "url":           f"https://reddit.com{c.get('permalink', '')}",
                                            "type":          "comment",
                                            "query_matched": query,
                                        })
                        except Exception as e:
                            log.warning(f"Comment fetch failed for {post_id}: {e}")

                # Paginate: move before_str_cur to oldest post timestamp
                if len(posts) < 100:
                    break
                oldest_ts = min(int(p.get("created_utc", before_ts)) for p in posts)
                before_str_cur = datetime.utcfromtimestamp(oldest_ts).strftime("%Y-%m-%dT%H:%M:%S")
                time.sleep(2)  # pause between pages

            log.info(f"     r/{subreddit}: {sub_post_count} new posts")
            time.sleep(2)  # pause between subreddits

        # Upload all records for this query
        if query_records:
            upload_batch_to_gcs(query_records, product_event)
            total_uploaded += len(query_records)
            log.info(f"  Uploaded {len(query_records)} records for query '{query}' -- total: {total_uploaded}")

        time.sleep(5)  # pause between queries

    log.info(f"\n  TOTAL: {total_uploaded} records for {product_event}")
    if total_uploaded < MIN_POST_THRESHOLD:
        log.warning(f"  WARNING: {total_uploaded} below threshold of {MIN_POST_THRESHOLD}")

    return total_uploaded

log.info("scrape_event_arctic_shift() defined")

2026-05-23 18:11:12,589 [INFO] scrape_event_arctic_shift() defined


In [17]:
# Run single event (testing)
test_event = events.iloc[0]
print(f"Test event: {test_event['product_name']} (launch: {test_event['launch_date'].date()})")
print(f"Subreddits: {test_event['pre_launch_subreddits']}")
print()

# Run with comments=False for a quick smoke test first
count = scrape_event(test_event, fetch_comments=True)
print(f"\n✓ Test complete — {count} records uploaded to GCS")

2026-05-07 14:56:33,962 [INFO] 
2026-05-07 14:56:33,963 [INFO] Scraping: fire_hd_10_(2019)
2026-05-07 14:56:33,964 [INFO] Window: 2019-08-01 → 2019-10-30
--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 50: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11

Test event: Fire HD 10 (2019) (launch: 2019-10-30)
Subreddits: r/kindlefire, r/tablets, r/android, r/gadgets



--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Use


✓ Test complete — 8 records uploaded to GCS


--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u26a0' in position 36: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Use

In [44]:
# ⚠ Run once to clear all checkpoint data — do not re-run
save_checkpoint({})
print("✓ Checkpoint cleared")

✓ Checkpoint cleared


In [31]:
# Override subreddits for underperforming events
fixes = {
    "iPhone 14": {
        "subreddits": ["iphone", "apple", "ios", "AppleWatch", 
                       "technology", "gadgets", "smartphones"],
        "query": "iPhone 14"
    },
    "Galaxy S22": {
        "subreddits": ["GalaxyS22", "samsung", "android", "GooglePixel",
                       "technology", "gadgets", "smartphones"],
        "query": "Galaxy S22"
    },
    "iPad mini 6": {
        "subreddits": ["ipad", "apple", "ios", "tablets", 
                       "gadgets", "technology"],
        "query": "iPad mini 6"
    },
    "MacBook Air M1": {
        "subreddits": ["apple", "mac", "macbook", "MacOS",
                       "technology", "gadgets", "laptops"],
        "query": "MacBook Air M1"
    },
    "Pixel 7": {
        "subreddits": ["GooglePixel", "pixel7", "android",
                       "technology", "gadgets", "smartphones"],
        "query": "Pixel 7"
    },
    "Pixel 5": {
        "subreddits": ["GooglePixel", "android", "technology", 
                       "gadgets", "smartphones"],
        "query": "Pixel 5"
    },
    "Galaxy S23": {
        "subreddits": ["GalaxyS23", "samsung", "android",
                       "technology", "gadgets", "smartphones"],
        "query": "Galaxy S23"
    },
    "iPhone 15": {
        "subreddits": ["iphone", "apple", "ios",
                       "technology", "gadgets", "smartphones"],
        "query": "iPhone 15"
    },
}

for product, config in fixes.items():
    mask = events["product_name"].str.contains(product, case=False)
    if mask.any():
        idx = events[mask].index[0]
        events.at[idx, "subreddits_list"] = config["subreddits"]
        events.at[idx, "product_name"] = config["query"]  # cleaner query
        print(f"✓ Fixed: {product}")

# Clear checkpoint for these events so Cell 9 re-scrapes them
checkpoint = load_checkpoint()
to_rerun = ["iphone_14", "galaxy_s22", "ipad_mini_6", "macbook_air_m1_"]
for key in to_rerun:
    if key in checkpoint:
        del checkpoint[key]
        print(f"✓ Cleared checkpoint: {key}")
save_checkpoint(checkpoint)

print("\nReady to re-run pipeline")

✓ Fixed: iPhone 14
✓ Fixed: Galaxy S22
✓ Fixed: iPad mini 6
✓ Fixed: MacBook Air M1
✓ Fixed: Pixel 7
✓ Fixed: Pixel 5
✓ Fixed: Galaxy S23
✓ Fixed: iPhone 15
✓ Cleared checkpoint: iphone_14
✓ Cleared checkpoint: ipad_mini_6
✓ Cleared checkpoint: macbook_air_m1_

Ready to re-run pipeline


In [16]:
# Verify GCS output
def list_bronze_files(product_event: str = None) -> pd.DataFrame:
    """List all files in GCS Bronze, optionally filtered by product event."""
    prefix = f"{BRONZE_PREFIX}/{product_event}/" if product_event else BRONZE_PREFIX
    blobs = list(gcs_bucket.list_blobs(prefix=prefix))

    rows = []
    for b in blobs:
        if b.name.endswith(".jsonl"):
            rows.append({
                "path":    b.name,
                "size_kb": round(b.size / 1024, 1),
                "updated": b.updated
            })

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["path", "size_kb", "updated"])


def preview_gcs_file(gcs_path: str, n: int = 5) -> pd.DataFrame:
    """Download and preview the first n records from a GCS JSONL file."""
    blob = gcs_bucket.blob(gcs_path)
    content = blob.download_as_text()
    lines = content.strip().split("\n")[:n]
    return pd.DataFrame([json.loads(l) for l in lines])


# Show what was uploaded for the test event
test_key = events.iloc[0]["product_name"].lower().replace(" ", "_")
files_df = list_bronze_files(test_key)
print(f"Files in GCS for '{test_key}':")
display(files_df)

if not files_df.empty:
    print("\nFirst 3 records:")
    display(preview_gcs_file(files_df.iloc[0]["path"], n=3))

Files in GCS for 'fire_hd_10_(2019)':


,path,size_kb,updated
0,bronze/reddit/fire_hd_10_(2019)/1778157808.jsonl,0.7,2026-05-07 12:43:28.982000+00:00
1,bronze/reddit/fire_hd_10_(2019)/1778158600.jsonl,4.6,2026-05-07 12:56:40.991000+00:00
2,bronze/reddit/fire_hd_10_(2019)/1778158652.jsonl,4.6,2026-05-07 12:57:33.011000+00:00
3,bronze/reddit/fire_hd_10_(2019)/1778167795.jsonl,4.6,2026-05-07 15:29:55.762000+00:00
4,bronze/reddit/fire_hd_10_(2019)/1778179588.jsonl,4.6,2026-05-07 18:46:29.714000+00:00
5,bronze/reddit/fire_hd_10_(2019)/1778182062.jsonl,4.6,2026-05-07 19:27:43.372000+00:00
6,bronze/reddit/fire_hd_10_(2019)/1778240598.jsonl,4.6,2026-05-08 11:43:18.630000+00:00
7,bronze/reddit/fire_hd_10_(2019)/1778244373.jsonl,4.6,2026-05-08 12:46:13.617000+00:00
8,bronze/reddit/fire_hd_10_(2019)/1778600710.jsonl,7.2,2026-05-12 15:45:10.990000+00:00



First 3 records:


,post_id,product_event,subreddit,title,body_text,upvotes,num_comments,created_utc,days_to_launch,url,type
0,cn3td9,fire_hd_10_(2019),kindlefire,Should I wait for a potential 2019 update on f...,"Part of me says to wait, but then again they ""...",8,12,1.565172e+09,83.58,https://reddit.com/r/kindlefire/comments/cn3td...,post


In [26]:
# Re-run iPhone 14 only
checkpoint = load_checkpoint()
del checkpoint["iphone_14"]
save_checkpoint(checkpoint)

In [32]:
#Run full pipeline (all events) - running this only after the single event test passes. Skips already completed events
checkpoint = load_checkpoint()
results = {}

for _, row in events.iterrows():
    event_key = row["product_name"].lower().replace(" ", "_")

    # Skip if already completed
    if event_key in checkpoint and checkpoint[event_key]["status"] == "complete":
        log.info(f"Skipping {event_key} — already in checkpoint")
        results[event_key] = checkpoint[event_key]["post_count"]
        continue

    try:
        count = scrape_event(row, fetch_comments=True)
        results[event_key] = count
        mark_complete(checkpoint, event_key, count)
    except Exception as e:
        log.error(f"Failed on {event_key}: {e}")
        results[event_key] = -1

    time.sleep(SLEEP_BETWEEN_EVENTS)

# Summary
print("\n" + "="*50)
print("SCRAPING COMPLETE — Summary")
print("="*50)
summary = pd.DataFrame([
    {"event": k, "records": v, "threshold_met": v >= MIN_POST_THRESHOLD}
    for k, v in results.items()
])
display(summary)
print(f"\nEvents meeting threshold (≥{MIN_POST_THRESHOLD}): "
      f"{summary['threshold_met'].sum()} / {len(summary)}")

2026-05-18 16:20:12,227 [INFO] Skipping fire_hd_10_(2019) — already in checkpoint
2026-05-18 16:20:12,235 [INFO] 
2026-05-18 16:20:12,236 [INFO] Scraping: fire_hd_8_(2020)_
2026-05-18 16:20:12,237 [INFO] Window: 2020-07-02 → 2020-09-30
2026-05-18 16:20:12,238 [INFO] Subreddits: ['kindlefire', 'amazonecho', 'amazon', 'tablets', 'android', 'gadgets', 'technology']
2026-05-18 16:20:12,239 [INFO]   → r/kindlefire
2026-05-18 16:20:24,907 [INFO]      r/kindlefire: 11 posts in window
2026-05-18 16:20:24,908 [INFO]   → r/amazonecho
2026-05-18 16:20:25,629 [INFO]      r/amazonecho: 0 posts in window
2026-05-18 16:20:25,630 [INFO]   → r/amazon
2026-05-18 16:20:26,300 [INFO]      r/amazon: 0 posts in window
2026-05-18 16:20:26,302 [INFO]   → r/tablets
2026-05-18 16:20:29,349 [INFO]      r/tablets: 0 posts in window
2026-05-18 16:20:29,350 [INFO]   → r/android
2026-05-18 16:20:31,503 [INFO]      r/android: 1 posts in window
2026-05-18 16:20:31,504 [INFO]   → r/gadgets
2026-05-18 16:20:32,204 [INFO


SCRAPING COMPLETE — Summary


,event,records,threshold_met
0,fire_hd_10_(2019),227,False
1,fire_hd_8_(2020)_,63,False
2,fire_7_(2019)_,5,False
3,ipad_pro_2020_,2834,True
4,ipad_air_2_(2014),5,False
5,macbook_air_m1,1307,True
6,ipad_mini_6,362,False
7,"macbook_pro_16""",1026,True
8,galaxy_s23,984,True
9,pixel_5,21112,True



Events meeting threshold (≥500): 12 / 35


In [38]:
# Testing 9b on 1 event
test_row = events[events["product_name"].str.contains("iPad Air 2", case=False)].iloc[0]
count = scrape_event_arctic_shift(test_row, fetch_comments=False)
print(f"Test result: {count} records")

2026-05-19 20:48:01,524 [INFO] 
2026-05-19 20:48:01,526 [INFO] Scraping (Arctic Shift): ipad_air_2_(2014)
2026-05-19 20:48:01,529 [INFO] Window: 2014-07-18 to 2014-10-16
2026-05-19 20:48:01,530 [INFO] Subreddits: ['ipad', 'apple', 'technology', 'gadgets']
2026-05-19 20:48:01,531 [INFO]   -> r/ipad
2026-05-19 20:48:02,570 [INFO]     status: 200
2026-05-19 20:48:02,571 [INFO]     posts returned: 0
2026-05-19 20:48:02,572 [INFO]      r/ipad: 0 posts
2026-05-19 20:48:02,573 [INFO]   -> r/apple
2026-05-19 20:48:03,209 [INFO]     status: 200
2026-05-19 20:48:03,211 [INFO]     posts returned: 2
2026-05-19 20:48:03,211 [INFO]      r/apple: 2 posts
2026-05-19 20:48:03,212 [INFO]   -> r/technology
2026-05-19 20:48:04,263 [INFO]     status: 200
2026-05-19 20:48:04,264 [INFO]     posts returned: 0
2026-05-19 20:48:04,265 [INFO]      r/technology: 0 posts
2026-05-19 20:48:04,266 [INFO]   -> r/gadgets
2026-05-19 20:48:04,967 [INFO]     status: 200
2026-05-19 20:48:04,968 [INFO]     posts returned: 0

Test result: 2 records


In [33]:
#Cell 9b - connects to 6b
# Re-run failing old events using browse mode
browse_targets = [
    "iPhone 13", "Galaxy S21", "Pixel 6", "MacBook Air M2", "iPad Pro",
    "MacBook Air M1", "iPhone 14", "iPad mini 6", "iPad Pro 2020",
    "Fire HD 10", "Fire HD 8", "Fire 7", "iPad Air 2",
    "MacBook Pro 16", "Galaxy S23", "Pixel 5", "iPhone 15", "Galaxy S22", "Pixel 7",
    "Samsung Galaxy S4", "Samsung Galaxy S5", "Samsung Galaxy S6", "Samsung Galaxy S7 Edge",
    "Samsung Galaxy S8", "Samsung Galaxy S20 FE", "Samsung Galaxy S21 Ultra", "Samsung Galaxy S22 Ultra",
    "Google Pixel 3a", "Google Pixel 4 XL", "Google Pixel 4a", "Moto G 3rd Gen", "Moto G 4th Gen", "Moto G Fast", "Moto G Power 2022",
    "LG G3"
]

checkpoint = load_checkpoint()
results_browse = {}

for _, row in events.iterrows():
    if not any(t.lower() in row["product_name"].lower() for t in browse_targets):
        continue

    event_key = row["product_name"].lower().replace(" ", "_")

    # Only re-run if below threshold
    if checkpoint.get(event_key, {}).get("post_count", 0) >= MIN_POST_THRESHOLD:
        log.info(f"Skipping {event_key} — already at threshold")
        continue

    # Clear this event from checkpoint so it re-uploads cleanly
    if event_key in checkpoint:
        del checkpoint[event_key]
    save_checkpoint(checkpoint)

    try:
        count = scrape_event_arctic_shift(row, fetch_comments=True)
        results_browse[event_key] = count
        mark_complete(checkpoint, event_key, count)
    except Exception as e:
        log.error(f"Failed on {event_key}: {e}")
        results_browse[event_key] = -1

    time.sleep(SLEEP_BETWEEN_EVENTS)

# Summary
print("\n" + "="*50)
print("BROWSE MODE — Summary")
print("="*50)
summary = pd.DataFrame([
    {"event": k, "records": v, "threshold_met": v >= MIN_POST_THRESHOLD}
    for k, v in results_browse.items()
])
display(summary)

2026-05-18 17:21:26,698 [INFO] 
2026-05-18 17:21:26,700 [INFO] Scraping (Arctic Shift): fire_hd_10_(2019)
2026-05-18 17:21:26,701 [INFO] Window: 2019-08-01 to 2019-10-30
2026-05-18 17:21:26,702 [INFO] Subreddits: ['kindlefire', 'amazonecho', 'amazon', 'tablets', 'android', 'gadgets', 'technology']
2026-05-18 17:21:26,703 [INFO]   -> r/kindlefire


2026-05-18 17:21:28,469 [INFO]     status: 200
2026-05-18 17:21:28,470 [INFO]     posts returned: 1
2026-05-18 17:21:29,445 [INFO]      r/kindlefire: 1 posts
2026-05-18 17:21:29,447 [INFO]   -> r/amazonecho
2026-05-18 17:21:30,795 [INFO]     status: 200
2026-05-18 17:21:30,797 [INFO]     posts returned: 0
2026-05-18 17:21:30,798 [INFO]      r/amazonecho: 0 posts
2026-05-18 17:21:30,799 [INFO]   -> r/amazon
2026-05-18 17:21:32,760 [INFO]     status: 200
2026-05-18 17:21:32,761 [INFO]     posts returned: 0
2026-05-18 17:21:32,762 [INFO]      r/amazon: 0 posts
2026-05-18 17:21:32,763 [INFO]   -> r/tablets
2026-05-18 17:21:35,078 [INFO]     status: 200
2026-05-18 17:21:35,079 [INFO]     posts returned: 0
2026-05-18 17:21:35,080 [INFO]      r/tablets: 0 posts
2026-05-18 17:21:35,081 [INFO]   -> r/android
2026-05-18 17:21:36,853 [INFO]     status: 200
2026-05-18 17:21:36,854 [INFO]     posts returned: 0
2026-05-18 17:21:36,855 [INFO]      r/android: 0 posts
2026-05-18 17:21:36,856 [INFO]   -


BROWSE MODE — Summary


,event,records,threshold_met
0,fire_hd_10_(2019),13,False
1,fire_hd_8_(2020)_,108,False
2,fire_7_(2019)_,5,False
3,ipad_air_2_(2014),5,False
4,ipad_mini_6,1007,True
5,iphone_14,14848,True
6,galaxy_s22,3610,True
7,samsung_galaxy_s4,2041,True
8,samsung_galaxy_s5,759,True
9,samsung_galaxy_s6,2328,True


In [ ]:
# Targeted re-scrape: Fire HD 10 and Fire HD 8 using Arctic Shift
fire_targets = {
    "Fire HD 10 (2019)": {
        "query_list": ["Fire HD 10"],
        "subreddits":  ["kindlefire", "amazonecho", "amazon", "tablets", "android", "gadgets", "technology"],
        "launch_date": "30/10/2019"
    },
    "Fire HD 8 (2020)": {
        "query_list": ["Fire HD 8"],
        "subreddits":  ["kindlefire", "amazonecho", "amazon", "tablets", "android", "gadgets", "technology"],
        "launch_date": "30/09/2020"
    }
}

results_fire = {}

for product_name, config in fire_targets.items():
    launch_dt    = pd.to_datetime(config["launch_date"], dayfirst=True, utc=True)
    window_start = launch_dt - timedelta(days=PRE_LAUNCH_DAYS)

    # Build a fake row matching what scrape_event_arctic_shift expects
    row = pd.Series({
        "product_name":    config["query"],   # clean query, no year bracket
        "launch_date":     launch_dt,
        "subreddits_list": config["subreddits"],
        "query_list":      config["query_list"],   # passes through to the function

    })

    # Clear checkpoint for this event
    event_key = product_name.lower().replace(" ", "_")
    checkpoint = load_checkpoint()
    if event_key in checkpoint:
        del checkpoint[event_key]
        save_checkpoint(checkpoint)
        log.info(f"Cleared checkpoint for {event_key}")

    log.info(f"Starting targeted scrape: {product_name}")
    log.info(f"Window: {window_start.date()} to {launch_dt.date()}")

    count = scrape_event_arctic_shift(row, fetch_comments=True)

    # Save under the original product key
    checkpoint = load_checkpoint()
    mark_complete(checkpoint, event_key, count)
    results_fire[product_name] = count

    time.sleep(SLEEP_BETWEEN_EVENTS)

print("\n" + "="*50)
print("Fire Tablets Scrape — Summary")
print("="*50)
for k, v in results_fire.items():
    status = "PASS" if v >= MIN_POST_THRESHOLD else "FAIL"
    print(f"  {k}: {v} records [{status}]")

2026-05-13 02:01:55,121 [INFO] Starting targeted scrape: Fire HD 10 (2019)
2026-05-13 02:01:55,124 [INFO] Window: 2019-08-01 to 2019-10-30
2026-05-13 02:01:55,125 [INFO] 
2026-05-13 02:01:55,127 [INFO] Scraping (Arctic Shift): fire_hd_10
2026-05-13 02:01:55,128 [INFO] Window: 2019-08-01 to 2019-10-30
2026-05-13 02:01:55,129 [INFO] Subreddits: ['kindlefire', 'amazonecho', 'amazon', 'tablets', 'android', 'gadgets', 'technology']
2026-05-13 02:01:55,130 [INFO]   -> r/kindlefire
2026-05-13 02:01:55,929 [INFO]     status: 200
2026-05-13 02:01:55,932 [INFO]     posts returned: 21
2026-05-13 02:02:09,677 [INFO]      r/kindlefire: 21 posts
2026-05-13 02:02:09,678 [INFO]   -> r/amazonecho
2026-05-13 02:02:10,261 [INFO]     status: 200
2026-05-13 02:02:10,262 [INFO]     posts returned: 0
2026-05-13 02:02:10,263 [INFO]      r/amazonecho: 0 posts
2026-05-13 02:02:10,264 [INFO]   -> r/amazon
2026-05-13 02:02:10,843 [INFO]     status: 200
2026-05-13 02:02:10,845 [INFO]     posts returned: 1
2026-05-


Fire Tablets Scrape — Summary
  Fire HD 10 (2019): 227 records [FAIL]
  Fire HD 8 (2020): 455 records [FAIL]


In [91]:
# Targeted re-scrape: Motorola and iPad
targets = {
    # ── Already in your list ─────────────────────────────────────
    "Acer Chromebook": {
        "query_list":       [ "Acer CB3", "Acer CB5"],
        "subreddits":  ["chromeos", "chromebook", "gadgets", "technology", "laptops", "tech"],
        "launch_date": "15/06/2015"
    },
    "iPad Pro 2018": {
        "query_list":       ["iPad Pro 11 inch", "iPad Pro 3rd generation"],
        "subreddits":  ["ipad", "apple", "technology", "tablets", "ios", "AppleLeaks"],
        "launch_date": "07/11/2018"
    },

    # ── Apple iPads ──────────────────────────────────────────────
    "iPad 7th Gen": {
        "query_list":       [ "iPad 2019", "iPad 10.2"],
        "subreddits":  ["ipad", "apple", "technology", "tablets", "ios"],
        "launch_date": "25/09/2019"
    },
    "iPad Pro M1": {
        "query_list":       [ "iPad Pro 2021", "iPad Pro 5th generation"],
        "subreddits":  ["ipad", "apple", "iPadPro", "technology", "tablets", "ios", "AppleLeaks"],
        "launch_date": "21/05/2021"
    },
    "iPad 9th Gen": {
        "query_list":       [ "iPad 2021", "iPad 10.2 2021"],
        "subreddits":  ["ipad", "apple", "technology", "tablets", "ios"],
        "launch_date": "24/09/2021"
    },
    "iPad Air M1": {
        "query_list":       [ "iPad Air 2022", "iPad Air 5th generation"],
        "subreddits":  ["ipad", "apple", "iPadPro", "technology", "tablets", "ios", "AppleLeaks"],
        "launch_date": "18/03/2022"
    },
    "iPad 10th Gen": {
        "query_list":       [ "iPad 2022", "iPad 10.9"],
        "subreddits":  ["ipad", "apple", "technology", "tablets", "ios", "AppleLeaks"],
        "launch_date": "26/10/2022"
    },

    # ── Amazon Fire tablets ──────────────────────────────────────
    "Fire HD 10 2021": {
        "query_list":       ["Fire HD 10 Plus"],
        "subreddits":  ["kindlefire", "amazon", "tablets", "android", "gadgets"],
        "launch_date": "26/10/2021"
    },

    # ── Google Pixel ─────────────────────────────────────────────
    "Pixel 3a": {
        "query_list":       ["Google Pixel 3a"],
        "subreddits":  ["GooglePixel", "pixel3a", "android", "technology", "gadgets"],
        "launch_date": "07/05/2019"
    },
    "Pixel 4 XL": {
        "query_list":       ["Google Pixel 4 XL"],
        "subreddits":  ["GooglePixel", "pixel4", "android", "technology", "gadgets"],
        "launch_date": "24/10/2019"
    },
    "Pixel 4a": {
        "query_list":       [ "Google Pixel 4a"],
        "subreddits":  ["GooglePixel", "pixel4a", "android", "technology", "gadgets"],
        "launch_date": "20/08/2020"
    },

    # ── Samsung Galaxy Tabs ──────────────────────────────────────
    "Galaxy Tab A 2016": {
        "query_list":       [ "Samsung Tab A 2016"],
        "subreddits":  ["GalaxyTab", "samsung", "android", "technology", "tablets"],
        "launch_date": "08/04/2016"
    },
    "Galaxy Tab A 2019": {
        "query_list":       [ "Samsung Tab A 10.1 2019"],
        "subreddits":  ["GalaxyTab", "samsung", "android", "technology", "tablets"],
        "launch_date": "26/04/2019"
    },
    "Galaxy Tab A8": {
        "query_list":       [ "Samsung Tab A8"],
        "subreddits":  ["GalaxyTab", "samsung", "android", "technology", "tablets"],
        "launch_date": "17/01/2022"
    },

    # ── Samsung Galaxy S ─────────────────────────────────────────
    "Galaxy S4": {
        "query_list":       ["Samsung Galaxy S4"],
        "subreddits":  ["galaxys4", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "27/04/2013"
    },
    "Galaxy S5": {
        "query_list":       [ "Samsung Galaxy S5"],
        "subreddits":  ["galaxys5", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "11/04/2014"
    },
    "Galaxy S6": {
        "query_list":       ["Samsung Galaxy S6"],
        "subreddits":  ["GalaxyS6", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "10/04/2015"
    },
    "Galaxy S7 Edge": {
        "query_list":       ["Samsung Galaxy S7 Edge"],
        "subreddits":  ["GalaxyS7", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "11/03/2016"
    },
    "Galaxy S8": {
        "query_list":       ["Samsung Galaxy S8"],
        "subreddits":  ["GalaxyS8", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "21/04/2017"
    },
    "Galaxy S20 FE": {
        "query_list":       ["Samsung Galaxy S20 FE"],
        "subreddits":  ["GalaxyS20FE", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "02/10/2020"
    },
    "Galaxy S21 Ultra": {
        "query_list":       ["Samsung Galaxy S21 Ultra"],
        "subreddits":  ["GalaxyS21", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "29/01/2021"
    },
    "Galaxy S22 Ultra": {
        "query_list":       [ "Samsung Galaxy S22 Ultra"],
        "subreddits":  ["GalaxyS22", "samsung", "android", "technology", "gadgets", "smartphones"],
        "launch_date": "25/02/2022"
    },

    # ── Asus / Microsoft ─────────────────────────────────────────
    "Asus Vivobook 15": {
        "query_list":       ["Asus VivoBook F512"],
        "subreddits":  ["ASUS", "laptops", "technology", "gadgets", "tech"],
        "launch_date": "01/09/2019"   # approximate — multiple variants
    },
    "Surface Pro 3": {
        "query_list":       ["Microsoft Surface Pro 3"],
        "subreddits":  ["Surface", "microsoft", "technology", "gadgets", "laptops", "tech"],
        "launch_date": "20/06/2014"
    },
}

results_fire = {}

for product_name, config in targets.items():
    launch_dt    = pd.to_datetime(config["launch_date"], dayfirst=True, utc=True)
    window_start = launch_dt - timedelta(days=PRE_LAUNCH_DAYS)

    # Build a fake row matching what scrape_event_arctic_shift expects
    row = pd.Series({
        "product_name":    event_key,   # clean query, no year bracket
        "launch_date":     launch_dt,
        "subreddits_list": config["subreddits"],
        "query_list":      config["query_list"],   # passes through to the function

    })

    # Clear checkpoint for this event
    event_key = product_name.lower().replace(" ", "_")
    checkpoint = load_checkpoint()
    if event_key in checkpoint:
        del checkpoint[event_key]
        save_checkpoint(checkpoint)
        log.info(f"Cleared checkpoint for {event_key}")

    log.info(f"Starting targeted scrape: {product_name}")
    log.info(f"Window: {window_start.date()} to {launch_dt.date()}")

    count = scrape_event_arctic_shift(row, fetch_comments=True)

    # Save under the original product key
    checkpoint = load_checkpoint()
    mark_complete(checkpoint, event_key, count)
    results_fire[product_name] = count

    time.sleep(SLEEP_BETWEEN_EVENTS)

print("\n" + "="*50)
print("Tablets Scrape — Summary")
print("="*50)
for k, v in results_fire.items():
    status = "PASS" if v >= MIN_POST_THRESHOLD else "FAIL"
    print(f"  {k}: {v} records [{status}]")

2026-05-23 18:41:56,655 [INFO] Starting targeted scrape: Acer Chromebook
2026-05-23 18:41:56,658 [INFO] Window: 2015-03-17 to 2015-06-15
2026-05-23 18:41:56,660 [INFO] 
2026-05-23 18:41:56,662 [INFO] Scraping (Arctic Shift): moto_g_4th_gen
2026-05-23 18:41:56,664 [INFO] Window: 2015-03-17 to 2015-06-15
2026-05-23 18:41:56,665 [INFO] Queries: ['Acer CB3', 'Acer CB5']
2026-05-23 18:41:56,690 [INFO] Subreddits: ['chromeos', 'chromebook', 'gadgets', 'technology', 'laptops', 'tech']
2026-05-23 18:41:56,692 [INFO] 
  Query: 'Acer CB3'
2026-05-23 18:41:56,694 [INFO]   -> r/chromeos
2026-05-23 18:42:12,509 [INFO]      r/chromeos: 13 new posts
2026-05-23 18:42:14,511 [INFO]   -> r/chromebook
2026-05-23 18:42:15,134 [INFO]      r/chromebook: 0 new posts
2026-05-23 18:42:17,136 [INFO]   -> r/gadgets
2026-05-23 18:42:18,282 [INFO]      r/gadgets: 0 new posts
2026-05-23 18:42:20,285 [INFO]   -> r/technology
2026-05-23 18:42:21,026 [INFO]      r/technology: 0 new posts
2026-05-23 18:42:23,028 [INFO]


Tablets Scrape — Summary
  Acer Chromebook: 120 records [FAIL]
  iPad Pro 2018: 150 records [FAIL]
  iPad 7th Gen: 1915 records [PASS]
  iPad Pro M1: 3870 records [PASS]
  iPad 9th Gen: 996 records [PASS]
  iPad Air M1: 254 records [FAIL]
  iPad 10th Gen: 821 records [PASS]
  Fire HD 10 2021: 71 records [FAIL]
  Pixel 3a: 1358 records [PASS]
  Pixel 4 XL: 2717 records [PASS]
  Pixel 4a: 5737 records [PASS]
  Galaxy Tab A 2016: 7 records [FAIL]
  Galaxy Tab A 2019: 0 records [FAIL]
  Galaxy Tab A8: 131 records [FAIL]
  Galaxy S4: 5065 records [PASS]
  Galaxy S5: 4925 records [PASS]
  Galaxy S6: 12394 records [PASS]
  Galaxy S7 Edge: 2942 records [PASS]
  Galaxy S8: 13846 records [PASS]
  Galaxy S20 FE: 632 records [PASS]
  Galaxy S21 Ultra: 2080 records [PASS]
  Galaxy S22 Ultra: 3007 records [PASS]
  Asus Vivobook 15: 2 records [FAIL]
  Surface Pro 3: 1217 records [PASS]


In [88]:
#Scrapping for Samsung tablets
targets = {
    "Galaxy Tab S2": {
        "query_list": [
            "Galaxy Tab S2",
            "Samsung Tab S2",
            "Tab S2 9.7",
            "Tab S2 8.0",
            "SM-T810",
            "SM-T710"
        ],
        "subreddits": [
            "GalaxyTab",
            "samsung",
            "Samsung",
            "SamsungGalaxy",
            "android",
            "technology",
            "tablets",
            "AndroidTablets",
            "pickanandroidforme",
            "hardware",
            "SamsungHelp",        # ← comma was missing before next entry
            "CheapAndroidTablets",
            "gadgets"
        ],
        "launch_date": "03/09/2015",
        "product_event": "galaxy_tab_s2",
        "posts_collected": 262,
        "posts_needed": 500
    },

    "Moto G Fast": {
        "query_list": [
            "Moto G Fast",
            "Motorola G Fast",
            "moto g fast 2020",
            "XT2045"
        ],
        "subreddits": [
            "motorola",
            "Motorola",
            "android",
            "AndroidQuestions",
            "pickanandroidforme",
            "budget_phones",
            "technology",
            "gadgets",
            "hardware",
            "nocontract"
        ],
        "launch_date": "18/06/2020",
        "product_event": "moto_g_fast",
        "posts_collected": 395,
        "posts_needed": 500
    },

    "Moto G 3rd Gen": {
        "query_list": [
            "Moto G 3rd gen",
            "Moto G 2015",
            "Motorola Moto G3",
            "XT1540",
            "XT1541",
            "XT1543",
            "Moto G3"
        ],
        "subreddits": [
            "motorola",
            "Motorola",
            "android",
            "AndroidQuestions",
            "pickanandroidforme",
            "budget_phones",
            "technology",
            "gadgets",
            "hardware",
            "nocontract"
        ],
        "launch_date": "28/07/2015",
        "product_event": "moto_g_3rd_gen",
        "posts_collected": 482,
        "posts_needed": 500
    },

    "Moto G 4th Gen": {
        "query_list": [
            "Moto G4",
            "Moto G 4th gen",
            "Moto G 2016",
            "Motorola Moto G4",
            "Moto G4 Plus",
            "Moto G4 Play",
            "XT1622",
            "XT1626",
            "XT1625"
        ],
        "subreddits": [
            "motorola",
            "Motorola",
            "android",
            "AndroidQuestions",
            "pickanandroidforme",
            "budget_phones",
            "technology",
            "gadgets",
            "hardware",
            "nocontract",
            "MotoG"
        ],
        "launch_date": "17/05/2016",
        "product_event": "moto_g_4th_gen",
        "posts_collected": 64,
        "posts_needed": 500
    }
}

results = {}

for product_name, config in targets.items():
    launch_dt    = pd.to_datetime(config["launch_date"], dayfirst=True, utc=True)
    window_start = launch_dt - timedelta(days=PRE_LAUNCH_DAYS)

    # Build a fake row matching what scrape_event_arctic_shift expects
    row = pd.Series({
        "product_name":    event_key,   # clean query, no year bracket
        "launch_date":     launch_dt,
        "subreddits_list": config["subreddits"],
        "query_list":      config["query_list"],   # passes through to the function
    })

    # Clear checkpoint for this event
    event_key = product_name.lower().replace(" ", "_")
    checkpoint = load_checkpoint()
    if event_key in checkpoint:
        del checkpoint[event_key]
        save_checkpoint(checkpoint)
        log.info(f"Cleared checkpoint for {event_key}")

    log.info(f"Starting targeted scrape: {product_name}")
    log.info(f"Window: {window_start.date()} to {launch_dt.date()}")

    count = scrape_event_arctic_shift(row, fetch_comments=True)

    # Save under the original product key
    checkpoint = load_checkpoint()
    mark_complete(checkpoint, event_key, count)
    results[product_name] = count

    time.sleep(SLEEP_BETWEEN_EVENTS)

print("\n" + "="*50)
print("Tablets Scrape — Summary")
print("="*50)
for k, v in results.items():
    status = "PASS" if v >= MIN_POST_THRESHOLD else "FAIL"
    print(f"  {k}: {v} records [{status}]")

2026-05-23 18:11:36,962 [INFO] Starting targeted scrape: Galaxy Tab S2
2026-05-23 18:11:36,963 [INFO] Window: 2015-06-05 to 2015-09-03
2026-05-23 18:11:36,964 [INFO] 
2026-05-23 18:11:36,965 [INFO] Scraping (Arctic Shift): galaxy_tab_s2
2026-05-23 18:11:36,966 [INFO] Window: 2015-06-05 to 2015-09-03
2026-05-23 18:11:36,967 [INFO] Queries: ['Galaxy Tab S2', 'Samsung Tab S2', 'Tab S2 9.7', 'Tab S2 8.0', 'SM-T810', 'SM-T710']
2026-05-23 18:11:36,968 [INFO] Subreddits: ['GalaxyTab', 'samsung', 'Samsung', 'SamsungGalaxy', 'android', 'technology', 'tablets', 'AndroidTablets', 'pickanandroidforme', 'hardware', 'SamsungHelp', 'CheapAndroidTablets', 'gadgets']
2026-05-23 18:11:36,969 [INFO] 
  Query: 'Galaxy Tab S2'
2026-05-23 18:11:36,969 [INFO]   -> r/GalaxyTab


2026-05-23 18:11:37,840 [INFO]      r/GalaxyTab: 8 new posts
2026-05-23 18:11:39,843 [INFO]   -> r/samsung
2026-05-23 18:11:45,431 [INFO]      r/samsung: 12 new posts
2026-05-23 18:11:47,431 [INFO]   -> r/Samsung
2026-05-23 18:11:48,019 [INFO]      r/Samsung: 0 new posts
2026-05-23 18:11:50,022 [INFO]   -> r/SamsungGalaxy
2026-05-23 18:11:51,698 [INFO]      r/SamsungGalaxy: 2 new posts
2026-05-23 18:11:53,700 [INFO]   -> r/android
2026-05-23 18:12:12,413 [INFO]      r/android: 27 new posts
2026-05-23 18:12:14,415 [INFO]   -> r/technology
2026-05-23 18:12:26,995 [INFO]      r/technology: 29 new posts
2026-05-23 18:12:28,996 [INFO]   -> r/tablets
2026-05-23 18:12:30,677 [INFO]      r/tablets: 3 new posts
2026-05-23 18:12:32,679 [INFO]   -> r/AndroidTablets
2026-05-23 18:12:34,261 [INFO]      r/AndroidTablets: 2 new posts
2026-05-23 18:12:36,263 [INFO]   -> r/pickanandroidforme
2026-05-23 18:12:36,855 [INFO]      r/pickanandroidforme: 0 new posts
2026-05-23 18:12:38,857 [INFO]   -> r/hard


Tablets Scrape — Summary
  Galaxy Tab S2: 517 records [PASS]
  Moto G Fast: 228 records [FAIL]
  Moto G 3rd Gen: 1693 records [PASS]
  Moto G 4th Gen: 836 records [PASS]


In [89]:
# Full GCS Inventory
all_files = list_bronze_files()
print(f"Total JSONL files in Bronze/Reddit: {len(all_files)}")
print(f"Total size: {all_files['size_kb'].sum():.1f} KB")
display(all_files)

Total JSONL files in Bronze/Reddit: 1396
Total size: 157676.8 KB


,path,size_kb,updated
0,bronze/reddit/acer_chromebook/1779307297.jsonl,3.7,2026-05-20 20:01:39.711000+00:00
1,bronze/reddit/acer_chromebook/1779372807.jsonl,116.2,2026-05-21 14:13:30.775000+00:00
2,bronze/reddit/acer_chromebook/1779372828.jsonl,120.5,2026-05-21 14:13:51.429000+00:00
3,bronze/reddit/acer_chromebook/1779372845.jsonl,107.0,2026-05-21 14:14:08.337000+00:00
4,bronze/reddit/acer_chromebook/1779372866.jsonl,77.3,2026-05-21 14:14:29.055000+00:00
...,...,...,...
1391,bronze/reddit/surface_pro_3/1779376489.jsonl,117.0,2026-05-21 15:14:52.722000+00:00
1392,bronze/reddit/surface_pro_3/1779376555.jsonl,101.4,2026-05-21 15:15:58.379000+00:00
1393,bronze/reddit/surface_pro_3/1779376586.jsonl,108.2,2026-05-21 15:16:29.311000+00:00
1394,bronze/reddit/surface_pro_3/1779376617.jsonl,138.4,2026-05-21 15:17:00.790000+00:00


In [67]:
# Run once — clears checkpoint so all events re-scrape
#save_checkpoint({})
# print("✓ Checkpoint cleared")

✓ Checkpoint cleared


In [90]:
# Record count per product event in GCS Bronze
from collections import defaultdict

all_blobs = list(gcs_bucket.list_blobs(prefix=BRONZE_PREFIX))
counts = defaultdict(int)

for blob in all_blobs:
    if not blob.name.endswith(".jsonl"):
        continue
    # Path format: bronze/reddit/{product_event}/{timestamp}.jsonl
    parts = blob.name.split("/")
    if len(parts) >= 3:
        product_event = parts[2]
        # Count lines = count records
        content = blob.download_as_text()
        line_count = len([l for l in content.strip().split("\n") if l])
        counts[product_event] += line_count

summary = pd.DataFrame([
    {
        "product_event": k,
        "records": v,
        "threshold_met": v >= MIN_POST_THRESHOLD
    }
    for k, v in sorted(counts.items(), key=lambda x: x[1], reverse=True)
])

print(f"Total events: {len(summary)}")
print(f"Meeting threshold (>={MIN_POST_THRESHOLD}): {summary['threshold_met'].sum()}")
print()
display(summary)

Total events: 63
Meeting threshold (>=500): 52



,product_event,records,threshold_met
0,pixel_6,40520,True
1,iphone_14,33638,True
2,pixel_5,21426,True
3,pixel_4a,18273,True
4,ipad_air_2_(2014),10455,True
...,...,...,...
58,motorola_g_3rd_generation,72,False
59,galaxy_tab_a_2016,42,False
60,ipad_9th_generation,33,False
61,asus_vivobook_15,22,False


In [64]:
# ── Consolidate duplicate GCS Bronze folders into canonical names ──

# Define canonical name → list of all folder names that should merge into it
consolidation_map = {
    "macbook_air_m1_(2020)": ["macbook_air_m1", "macbook_air_m1_"],
    "fire_hd_8_(2020)":      ["fire_hd_8_(2020)_", "fire_hd_8"],
    "fire_hd_10_(2019)":     ["fire_hd_10_(2019)", "fire_hd_10"],
    "ipad_air_2_(2014)":     ["ipad_air_2", "ipad_air_2_(2014)"],
    "moto_g_4th_gen":      ["moto_g_4th_gen", "moto_g_4th_generation", "moto_g4", "moto_g_4th_gen\""],
    "moto_g_3rd_gen":      ["moto_g_3rd_gen", "moto_g_3rd_generation", "moto_g_2015"],
}

for canonical, sources in consolidation_map.items():
    print(f"\nConsolidating into: {canonical}")
    moved = 0
    deleted = 0

    for source in sources:
        prefix = f"{BRONZE_PREFIX}/{source}/"
        blobs = list(gcs_bucket.list_blobs(prefix=prefix))
        if not blobs:
            print(f"  {source}: no files found")
            continue

        for blob in blobs:
            if not blob.name.endswith(".jsonl"):
                continue

            # Build new path under canonical folder
            filename = blob.name.split("/")[-1]
            new_path = f"{BRONZE_PREFIX}/{canonical}/{source}_{filename}"

            # Copy to canonical folder
            gcs_bucket.copy_blob(blob, gcs_bucket, new_path)
            moved += 1

            # Delete original only if it's not already in the canonical folder
            if source != canonical:
                blob.delete()
                deleted += 1

        print(f"  {source}: moved {len(blobs)} files")

    print(f"  Total moved: {moved}, deleted: {deleted}")

# Verify result
print("\n\nFinal record counts after consolidation:")
from collections import defaultdict
counts = defaultdict(int)
for blob in gcs_bucket.list_blobs(prefix=BRONZE_PREFIX):
    if not blob.name.endswith(".jsonl"):
        continue
    parts = blob.name.split("/")
    if len(parts) >= 3:
        counts[parts[2]] += len(blob.download_as_text().strip().split("\n"))

summary = pd.DataFrame([
    {"product_event": k, "records": v, "threshold_met": v >= MIN_POST_THRESHOLD}
    for k, v in sorted(counts.items(), key=lambda x: x[1], reverse=True)
])
display(summary)


Consolidating into: macbook_air_m1_(2020)
  macbook_air_m1: no files found
  macbook_air_m1_: no files found
  Total moved: 0, deleted: 0

Consolidating into: fire_hd_8_(2020)
  fire_hd_8_(2020)_: no files found
  fire_hd_8: no files found
  Total moved: 0, deleted: 0

Consolidating into: fire_hd_10_(2019)
  fire_hd_10_(2019): moved 81 files
  fire_hd_10: no files found
  Total moved: 81, deleted: 0

Consolidating into: ipad_air_2_(2014)
  ipad_air_2: no files found
  ipad_air_2_(2014): moved 56 files
  Total moved: 56, deleted: 0

Consolidating into: moto_g_4th_gen
  moto_g_4th_gen: moved 8 files
  moto_g_4th_generation: no files found
  moto_g4: moved 4 files
  moto_g_4th_gen": no files found
  Total moved: 12, deleted: 4

Consolidating into: moto_g_3rd_gen
  moto_g_3rd_gen: moved 4 files
  moto_g_3rd_generation: no files found
  moto_g_2015: no files found
  Total moved: 4, deleted: 0


Final record counts after consolidation:


,product_event,records,threshold_met
0,pixel_6,40520,True
1,iphone_14,33638,True
2,pixel_5,21426,True
3,ipad_air_2_(2014),10455,True
4,iphone_13,9751,True
...,...,...,...
57,motorola_g_3rd_generation,72,False
58,ipad_9th_generation,33,False
59,asus_vivobook_15,22,False
60,fire_7_(2019)_,10,False
